# EAGF Notebook 4: Pareto-Front MOO Visualisation

This notebook demonstrates and visualises the multi-objective optimisation (MOO)
used in EAGF (Paper Section 3.7):
- Sweep of the 5×5 (lambda_RP × lambda_C) Lagrangian grid
- Pareto-front identification (non-dominated sorting)
- Privacy–Fairness trade-off surface
- Selection of the best-TI model from the Pareto front

**Note:** Full 25-run grid is run here with fewer epochs for speed. Use
`python run_eagf.py --epochs 50` for paper-quality results.

[![GitHub](https://img.shields.io/badge/GitHub-aliakarma%2Feagf-blue?logo=github)](https://github.com/aliakarma/eagf)  **Repository:** [https://github.com/aliakarma/eagf](https://github.com/aliakarma/eagf)

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/aliakarma/eagf.git"
REPO_DIR_NAME = "eagf"

def find_project_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "configs").exists() and (candidate / "src").exists():
            return candidate
    return None

PROJECT_ROOT_PATH = find_project_root(Path.cwd())

if PROJECT_ROOT_PATH is None:
    clone_target = Path.cwd() / REPO_DIR_NAME
    if not clone_target.exists():
        print(f"Cloning repository into {clone_target}...")
        subprocess.run(["git", "clone", REPO_URL, str(clone_target)], check=True)
    PROJECT_ROOT_PATH = find_project_root(clone_target)
    if PROJECT_ROOT_PATH is None:
        raise RuntimeError("Could not locate project root after cloning.")
    os.chdir(PROJECT_ROOT_PATH)

PROJECT_ROOT = str(PROJECT_ROOT_PATH.resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Using PROJECT_ROOT={PROJECT_ROOT}")

### Understanding the Tools: Importing Libraries
This cell imports essential Python libraries that provide powerful functionalities for data manipulation, numerical operations, plotting, and building neural networks.
*   `numpy` is for numerical computations.
*   `pandas` is for data analysis and manipulation.
*   `matplotlib.pyplot` is for creating static, interactive, and animated visualizations.
*   `torch`, `torch.nn`, and `torch.optim` are from PyTorch, a popular framework for deep learning, used here to define and train neural network models.
*   `%matplotlib inline` is a special command for Jupyter/Colab notebooks to display plots directly within the output cells.

In [ ]:
import os
import sys
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

PROJECT_ROOT = Path(PROJECT_ROOT) if 'PROJECT_ROOT' in globals() else Path.cwd()
if not (PROJECT_ROOT / 'configs').exists() and (PROJECT_ROOT.parent / 'configs').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = str(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.training.pareto_trainer import run_pareto_search
from src.utils.data_loader import generate_demo_biometric

print(f'PROJECT_ROOT={PROJECT_ROOT}')

### Setting Up Optimization Parameters
This cell defines the range of `lambda` values that control the optimization process. These 'lambda' parameters are crucial for balancing different objectives (like fairness and privacy) in our multi-objective optimization (MOO).
*   `lambda_RP_values` and `lambda_C_values` are arrays of values for `lambda_RP` (Recall Parity) and `lambda_C` (Clarity), spanning from 0.001 to 10 on a logarithmic scale. These will be used to explore different trade-offs.
*   `lambda_P` (Privacy) is kept constant at 1.0 for this demonstration.

In [ ]:
lambda_RP_values = np.logspace(-3, 1, 5)
lambda_C_values  = np.logspace(-3, 1, 5)
lambda_P = 1.0

print("Lambda_RP:", lambda_RP_values)
print("Lambda_C:", lambda_C_values)

### Load Configuration and Demo Dataset
This cell loads the project configuration and prepares a demo biometric dataset used by the Pareto trainer.

In [ ]:
with open(os.path.join(PROJECT_ROOT, 'configs', 'biometric_default.yaml')) as f:
    cfg = yaml.safe_load(f)

demo_dataset = generate_demo_biometric(n_samples=1600, seed=42)
cfg.setdefault('governance', {})
cfg['governance']['pareto_grid_size'] = 5

### Using Real Metrics from Training Runs
This notebook now reads metrics from actual training runs produced by the EAGF trainer and Pareto search pipeline.
- `C`, `RP`, `P`, `A`, and `TI` are taken from run outputs, not sampled from random distributions.
- `TI` is the trainer-computed Trust Index and includes the accountability pillar from each run.
- The DataFrame used for plots is built from `run_pareto_search(...)["all_results"]`.

In [ ]:
def to_plot_metrics(entry):
    return {
        'C': float(entry['clarity']),
        'RP': float(entry['recall_parity']),
        'P': float(entry['privacy']),
        'A': float(entry['accountability']),
        'TI': float(entry['trust_index']),
        'Acc': float(entry['accuracy']),
    }

### Run Pareto Grid Search
This cell executes the actual 5×5 Pareto sweep via `run_pareto_search`, producing model-derived metrics for each `(lambda_rp, lambda_c)` combination.

In [ ]:
pareto_output_dir = os.path.join(PROJECT_ROOT, 'results', 'notebook_pareto')
os.makedirs(pareto_output_dir, exist_ok=True)

pareto_results = run_pareto_search(
    config=cfg,
    lambda_rp_range=(1e-3, 1e1),
    lambda_c_range=(1e-3, 1e1),
    n_steps=5,
    seed=42,
    device='cpu',
    output_dir=pareto_output_dir,
    dataset=demo_dataset,
    )

### Build Plot Table from Pareto Results
This cell transforms each Pareto run result into plotting columns (`C`, `RP`, `P`, `A`, `TI`, `Acc`) so downstream analysis and figures use model-derived values.

In [ ]:
results = []
for row in pareto_results['all_results']:
    m = to_plot_metrics(row)
    results.append({
        'lambda_rp': float(row['lambda_rp']),
        'lambda_c': float(row['lambda_c']),
        'C': m['C'],
        'RP': m['RP'],
        'P': m['P'],
        'A': m['A'],
        'TI': m['TI'],
        'Acc': m['Acc'],
    })

df = pd.DataFrame(results)
df.head()

### Summary Statistics
The table below reports variability and descriptive statistics from the real Pareto-grid outputs.

In [ ]:
print('STD CHECK:')
print(df[['C', 'RP', 'P', 'A', 'TI']].std())

print('\nSUMMARY:')
print(df.describe())

### Analyzing Metric Statistics
This cell provides a quick statistical overview of the metrics collected during the grid search. It helps us understand the variability and distribution of the Fairness (C), Recall Parity (RP), Privacy (P), and Trust Index (TI) across the different `lambda` configurations.
*   `df[['C', 'RP', 'P', 'TI']].std()` calculates the standard deviation for the specified metrics, showing how much they vary.
*   `df.describe()` generates descriptive statistics (count, mean, standard deviation, min, max, quartiles) for all numerical columns in the DataFrame, offering a comprehensive summary.

In [ ]:
print("STD CHECK:")
print(df[["C", "RP", "P", "TI"]].std())

print("\nSUMMARY:")
print(df.describe())

### Identifying the Pareto Front
This cell implements the `get_pareto_front` function, which identifies the "Pareto front" from the collected results. The Pareto front consists of solutions where no objective can be improved without sacrificing another. These are the optimal trade-off points.
*   The function iterates through each solution and checks if it's dominated by any other solution (i.e., if another solution is better or equal in all objectives and strictly better in at least one).
*   Solutions that are not dominated are considered Pareto-optimal and are included in the `pareto_df` DataFrame.

In [ ]:
def get_pareto_front(df):
    pareto = []

    for i, row in df.iterrows():
        dominated = False

        for j, other in df.iterrows():
            if (
                (other["P"] >= row["P"]) and
                (other["RP"] >= row["RP"]) and
                ((other["P"] > row["P"]) or (other["RP"] > row["RP"]))
            ):
                dominated = True
                break

        if not dominated:
            pareto.append(row)

    return pd.DataFrame(pareto)

pareto_df = get_pareto_front(df)

### Visualizing the Pareto Trade-off
This cell generates a scatter plot to visualize the trade-off between Privacy (P) and Recall Parity (RP), highlighting the Pareto front. This plot is essential for understanding the non-dominated solutions.
*   All collected data points are shown, colored by their Trust Index (TI).
*   Solutions on the `pareto_df` (the Pareto front) are specifically marked in red, making them easy to identify.
*   The `best` model (the one with the highest Trust Index) is marked with a black star, indicating a preferred operating point on the front.
*   The plot helps to visually analyze how improving privacy might impact fairness and vice-versa.
*   The plot is saved as `/tmp/pareto_tradeoff.png` with 300 DPI.

In [ ]:
plt.figure(figsize=(8,6))

# All points
scatter = plt.scatter(df["P"], df["RP"], c=df["TI"], cmap="viridis")

# Pareto front
plt.scatter(pareto_df["P"], pareto_df["RP"],
            color="red", s=100, label="Pareto front")

# Best point
best = df.loc[df["TI"].idxmax()]
plt.scatter(best["P"], best["RP"],
            color="black", s=150, marker="*", label="Best TI")

plt.xlabel("Privacy (P)")
plt.ylabel("Recall Parity (RP)")
plt.title("Pareto Trade-off: Privacy vs Fairness")

plt.colorbar(scatter, label="Trust Index (TI)")
plt.legend()
plt.grid(True)

out_tradeoff = os.path.join(PROJECT_ROOT, 'figures', 'notebook4_pareto_tradeoff.png')
plt.savefig(out_tradeoff, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out_tradeoff}')

### Visualizing Trust Index Heatmap
This cell generates a heatmap that shows how the Trust Index (TI) varies across different combinations of `lambda_rp` and `lambda_c` values. This helps in understanding which `lambda` settings lead to better overall performance.
*   `df.pivot` reshapes the DataFrame to easily create a grid where `lambda_rp` and `lambda_c` are the axes and `TI` is the value in the grid.
*   `plt.imshow` creates the heatmap, with color intensity representing the Trust Index.
*   The axes are labeled with the `lambda` values, providing a clear map of the optimization landscape.
*   The plot is saved as `/tmp/trust_index_heatmap.png` with 300 DPI.

In [ ]:
pivot = df.pivot(index="lambda_rp", columns="lambda_c", values="TI")

plt.figure(figsize=(6,5))
plt.imshow(pivot.values, aspect="auto", origin="lower")

plt.colorbar(label="TI")

plt.xticks(range(len(lambda_C_values)), [f"{x:.3f}" for x in lambda_C_values])
plt.yticks(range(len(lambda_RP_values)), [f"{x:.3f}" for x in lambda_RP_values])

plt.xlabel("lambda_C")
plt.ylabel("lambda_RP")
plt.title("Trust Index Heatmap")

out_heatmap = os.path.join(PROJECT_ROOT, 'figures', 'notebook4_trust_index_heatmap.png')
plt.savefig(out_heatmap, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out_heatmap}')